In [3]:
#Importar bibiotecas
import io
from pathlib import Path
import pandas as pd

#definir rutas de directorios
RAW_DIR = Path.cwd().parent.parent / 'data' / 'data' / 'raw'
PROCESSED_DIR = Path.cwd().parent.parent / 'data' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Leer el archivo en bytes, le quita el BOM si lo tiene, y lo decodifica como latin-1 para que pandas lo lea sin errores de codificación.
def leer_texto_limpio(ruta):
    raw = ruta.read_bytes()
    if raw.startswith(b'\xef\xbb\xbf'):
        # tiene BOM -> el archivo SÍ está en UTF-8 de verdad; utf-8-sig quita el BOM solo
        return io.StringIO(raw.decode('utf-8-sig'))
    # sin BOM -> estos exports del INE vienen en latin-1
    return io.StringIO(raw.decode('latin-1'))

#para cargar los datos de población de Gipuzkoa desde un archivo CSV, 
def cargar_poblacion_gipuzkoa(nombre_archivo, nombre_variable, nombre_salida):
    #leer csv de forma correcta
    df = pd.read_csv(
        leer_texto_limpio(RAW_DIR / nombre_archivo),
        sep='\t',
        thousands='.',
        decimal=',',
    )
    #Cambiar nombres de columnas y eliminar columna innecesariaa
    df = df.rename(columns={
        'Municipios': 'municipio_raw',
        'Secciones': 'seccion_raw',
        'Sexo': 'sexo',
        nombre_variable: 'variable',
        'Periodo': 'anio',
        'Total': 'poblacion',
    })
    df = df.drop(columns=['Provincias'])

   # la columna que sobra después de renombrar las fijas es la variable propia de esta tabla
    columnas_fijas = {'municipio_raw', 'seccion_raw', 'sexo', 'anio', 'poblacion'}
    columna_variable = [c for c in df.columns if c not in columnas_fijas][0]
    df = df.rename(columns={columna_variable: 'variable'})

    # Clasifica cada fila como provincia, municipio o sección según qué columnas de geografía vienen vacías, y guarda el resultado en una nueva columna 'nivel'
    def nivel(row):
        if pd.isna(row['municipio_raw']):
            return 'provincia'
        elif pd.isna(row['seccion_raw']):
            return 'municipio'
        return 'seccion'

    df['nivel'] = df.apply(nivel, axis=1)

    #Divide la columna 'municipio_raw' en dos nuevas columnas: 'municipio_codigo' y 'municipio_nombre', usando una expresión regular para extraer el código y el nombre del municipio.
    muni_split = df['municipio_raw'].str.extract(r'^(\d+)\s+(.*)$')
    df['municipio_codigo'] = muni_split[0]
    df['municipio_nombre'] = muni_split[1]

    df.to_csv(PROCESSED_DIR / nombre_salida, index=False)
    return df

In [9]:
#Cargar los 4 csv de población de Gipuzkoa, cada uno con su variable propia, y guardarlos en archivos procesados
df1 = cargar_poblacion_gipuzkoa('Población1.csv', 'Nacionalidad', 'poblacion1_nacionalidad.csv')
df2 = cargar_poblacion_gipuzkoa('Población2.csv', 'Lugar de nacimiento', 'poblacion2_pais_nacimiento_simple.csv')
df3 = cargar_poblacion_gipuzkoa('Población3.csv', 'Relación entre lugar de nacimiento y lugar de residencia', 'poblacion3_relacion_nacimiento_residencia.csv')
df4 = cargar_poblacion_gipuzkoa('Población4.csv', 'País de nacimiento', 'poblacion4_pais_nacimiento_detalle.csv')

# Etiquetar cada tabla antes de unirlas, para saber después de dónde viene cada fila
df1['tabla'] = 'nacionalidad'
df2['tabla'] = 'pais_nacimiento_simple'
df3['tabla'] = 'relacion_nacimiento_residencia'
df4['tabla'] = 'pais_nacimiento_detalle'

# pd.concat apila los 4 DataFrames uno debajo de otro (funciona porque tienen las mismas columnas)
poblacion = pd.concat([df1, df2, df3, df4], ignore_index=True)

poblacion.to_csv(PROCESSED_DIR / 'poblacion_gipuzkoa_combinada.csv', index=False)
poblacion.head()

,municipio_raw,seccion_raw,sexo,variable,anio,poblacion,nivel,municipio_codigo,municipio_nombre,tabla
0,NaN,NaN,Hombres,Española,2025,320128.0,provincia,NaN,NaN,nacionalidad
1,NaN,NaN,Hombres,Española,2024,320800.0,provincia,NaN,NaN,nacionalidad
2,NaN,NaN,Hombres,Española,2023,321629.0,provincia,NaN,NaN,nacionalidad
3,NaN,NaN,Hombres,Española,2022,323058.0,provincia,NaN,NaN,nacionalidad
4,NaN,NaN,Hombres,Española,2021,324345.0,provincia,NaN,NaN,nacionalidad


In [ ]:
poblacion.info()
#valores nulos y duplicados
print(poblacion.isna().sum())
print(poblacion.duplicated().sum())

# valores únicos de sexo
print(poblacion['sexo'].unique())

# valores únicos de 'variable', tabla por tabla (cada una tiene categorías distintas)
for t in poblacion['tabla'].unique():
    print(t, '->', poblacion[poblacion['tabla'] == t]['variable'].unique())
    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 283465 entries, 0 to 283464
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   municipio_raw     283020 non-null  object 
 1   seccion_raw       243860 non-null  object 
 2   sexo              283465 non-null  object 
 3   variable          283465 non-null  object 
 4   anio              283465 non-null  int64  
 5   poblacion         282219 non-null  float64
 6   nivel             283465 non-null  object 
 7   municipio_codigo  283020 non-null  object 
 8   municipio_nombre  283020 non-null  object 
 9   tabla             283465 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 21.6+ MB
municipio_raw         445
seccion_raw         39605
sexo                    0
variable                0
anio                    0
poblacion            1246
nivel                   0
municipio_codigo      445
municipio_nombre      445
tabla           

In [7]:
nulos = poblacion[poblacion['poblacion'].isna()]
print(nulos['nivel'].value_counts())
print(nulos['tabla'].value_counts())

nivel
seccion    1246
Name: count, dtype: int64
tabla
pais_nacimiento_detalle           882
relacion_nacimiento_residencia    252
nacionalidad                       56
pais_nacimiento_simple             56
Name: count, dtype: int64
